# 02. 윈도우 임베딩 유사도로 경계 후보 탐색

**2026-10-21 / hyungjin**

01번에서 저장한 **문자 길이 기준 오버랩 윈도우 임베딩**을 읽고, 인접 윈도우의 코사인 유사도가 낮아지는 곳을 경계 후보로 찾습니다. 모델이나 API를 다시 호출하지 않습니다.

후보 선택 방법은 `threshold`, `quantile`, `top_k`, `local_minima`, `depth`의 다섯 가지입니다. 모든 방법을 비교한 뒤 `SELECTED_METHOD`로 사용할 결과를 고릅니다. 후보는 아직 확정된 주제 경계가 아닙니다.

## 1. 환경과 입력 선택

필요하면 아래 설치 명령의 주석을 해제하세요. 01번을 먼저 실행해 `outputs/<실행폴더>/` 결과를 만들어야 합니다. `EMBEDDING_RUN_DIR=None`이면 가장 최근 임베딩 결과를 선택하고 경로와 모델을 표시합니다. 비교 실험에서는 경로를 직접 지정하는 편이 명확합니다.

부분 입력 결과도 분석할 수 있지만, 그 결과가 전체 회의에 대한 분석은 아니라는 점을 표시합니다. 윈도우가 하나뿐이면 인접 비교가 불가능하므로 01번에서 입력 범위를 늘리세요.

In [ ]:
# %pip install numpy pandas matplotlib

from pathlib import Path
from datetime import datetime, timezone, timedelta
from importlib.metadata import version, PackageNotFoundError
import hashlib
import json
import os
import re
import time
import uuid

import numpy as np
import pandas as pd
from IPython.display import display


def find_experiment_dir():
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base, base / "playground/hyungjin/embedding-test",
                          base / "remeet/playground/hyungjin/embedding-test"):
            if (candidate / "01_text_embeddings.ipynb").is_file() and (candidate.parent / "test/input").is_dir():
                return candidate.resolve()
    raise FileNotFoundError("embedding-test 폴더 또는 remeet 폴더에서 커널을 실행하세요.")


EXPERIMENT_DIR = find_experiment_dir()
OUTPUT_ROOT = EXPERIMENT_DIR / "outputs"
EMBEDDING_RUN_DIR = None  # 예: OUTPUT_ROOT / "20261021-120000-huggingface-ab123456"
available_runs = sorted(p.parent for p in OUTPUT_ROOT.glob("*/manifest.json")
                        if (p.parent / "embeddings.npz").is_file())
print("임베딩 결과 폴더:", [p.name for p in available_runs])
if EMBEDDING_RUN_DIR is None:
    if not available_runs:
        raise FileNotFoundError("임베딩 결과가 없습니다. 01번 노트북을 먼저 실행하세요.")
    EMBEDDING_RUN_DIR = available_runs[-1]
else:
    EMBEDDING_RUN_DIR = Path(EMBEDDING_RUN_DIR)
    if not EMBEDDING_RUN_DIR.is_absolute():
        EMBEDDING_RUN_DIR = EXPERIMENT_DIR / EMBEDDING_RUN_DIR
print("선택한 입력:", EMBEDDING_RUN_DIR)

## 2. 저장 결과의 정합성 확인

입력 해시, 윈도우 순서, 문자 범위, 행렬 크기와 정규화를 검증합니다. 서로 다른 모델의 벡터를 섞지 않고 **한 실행 결과 안에서** 유사도를 비교합니다. 문자 위치는 01번의 `source.txt` 기준이며 끝 위치는 포함하지 않습니다.

In [ ]:
def load_embedding_run(run_dir):
    run_dir = Path(run_dir)
    manifest = json.loads((run_dir / "manifest.json").read_text(encoding="utf-8"))
    if manifest.get("schema_version") != 3 or manifest.get("window_unit") != "unicode_character":
        raise ValueError("01번의 문자 길이 방식(schema_version=3) 결과가 필요합니다.")
    source = (run_dir / "source.txt").read_text(encoding="utf-8")
    if hashlib.sha256(source.encode("utf-8")).hexdigest() != manifest["normalized_source_sha256"]:
        raise ValueError("source.txt 해시가 manifest와 다릅니다.")
    windows = [json.loads(line) for line in (run_dir / "windows.jsonl").read_text(encoding="utf-8").splitlines()]
    blocks = [json.loads(line) for line in (run_dir / "blocks.jsonl").read_text(encoding="utf-8").splitlines()]
    with np.load(run_dir / "embeddings.npz", allow_pickle=False) as archive:
        vectors = archive["embeddings"]
    if (vectors.ndim != 2 or list(vectors.shape) != manifest["embedding_shape"]
            or len(windows) != vectors.shape[0] or len(windows) != manifest["window_count"]):
        raise ValueError("윈도우와 임베딩 행 수/차원이 일치하지 않습니다.")
    if len(windows) < 2:
        raise ValueError("인접 유사도 비교에는 윈도우가 2개 이상 필요합니다.")
    if not blocks or len(blocks) != manifest["selected_block_count"]:
        raise ValueError("선택한 원문 블록 정보가 일치하지 않습니다.")
    for i, w in enumerate(windows):
        start, end = w["start_char"], w["end_char"]
        if (w["window_index"] != i or not 0 <= start < end <= len(source)
                or source[start:end] != w["text"] or w["char_count"] != end - start
                or end - start > manifest["window_chars"]):
            raise ValueError(f"윈도우 {i}의 원문 위치나 텍스트가 일치하지 않습니다.")
        if i:
            prev = windows[i - 1]
            if (start - prev["start_char"] != manifest["stride_chars"]
                    or prev["end_char"] - start != manifest["overlap_chars"]):
                raise ValueError("저장한 윈도우의 이동 간격/오버랩이 다릅니다.")
        if i < len(windows) - 1 and end - start != manifest["window_chars"]:
            raise ValueError("마지막 이외의 윈도우 길이가 일정하지 않습니다.")
    if (windows[0]["start_char"] != blocks[0]["start_char"]
            or windows[-1]["end_char"] != blocks[-1]["end_char"]):
        raise ValueError("윈도우가 선택한 본문 전체를 덮지 않습니다.")
    if not np.isfinite(vectors).all() or not np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-5):
        raise ValueError("임베딩에 비정상 값이 있거나 L2 정규화되지 않았습니다.")
    return manifest, source, windows, blocks, vectors


manifest, source_text, windows, blocks, embeddings = load_embedding_run(EMBEDDING_RUN_DIR)
body_start, body_end = windows[0]["start_char"], windows[-1]["end_char"]
print("임베딩 모델:", manifest["embedding_info"]["model"])
print("윈도우 수/차원:", embeddings.shape)
print("부분 입력 결과:", manifest["is_partial"])
print("윈도우/오버랩:", manifest["window_chars"], manifest["overlap_chars"], "문자")

## 3. 유사도와 후보 위치 계산

L2 정규화된 벡터 `e[i]`와 `e[i+1]`의 내적이 코사인 유사도입니다. 여기서는 전체 쌍 행렬 대신 **인접 쌍만 계산**하므로 회의 길이에 비례해 처리합니다.

오버랩이 있으면 의미가 바뀌어도 공유 텍스트 때문에 유사도가 높을 수 있습니다. 비교점의 위치는 **왼쪽 윈도우 끝과 오른쪽 윈도우 시작의 중간**으로 잡습니다. 이는 겹침 구간 안의 대표 위치이며 LLM이 확인한 정확한 경계가 아닙니다. `region_start_char:region_end_char`는 그 겹침 구간입니다. 오버랩이 없으면 두 값이 같은 정확한 접점입니다.

`depth`는 주변 좌우 최대 유사도와 현재 유사도의 차이를 합한 값입니다. 주변보다 깊게 내려간 골짜기를 찾는 보조 점수이며 통계적 신뢰도가 아닙니다.

In [ ]:
def adjacent_scores(vectors, windows, depth_radius=3):
    if type(depth_radius) is not int or depth_radius < 1:
        raise ValueError("depth_radius는 양의 정수여야 합니다.")
    similarities = np.clip(np.sum(vectors[:-1] * vectors[1:], axis=1), -1.0, 1.0)
    rows = []
    for i, value in enumerate(similarities):
        left_peak = float(max(similarities[max(0, i - depth_radius):i + 1]))
        right_peak = float(max(similarities[i:min(len(similarities), i + depth_radius + 1)]))
        region_start, region_end = windows[i + 1]["start_char"], windows[i]["end_char"]
        rows.append({"pair_index": i, "left_window_index": i, "right_window_index": i + 1,
                     "candidate_char": (region_start + region_end) // 2,
                     "region_start_char": region_start, "region_end_char": region_end,
                     "similarity": float(value), "depth": max(0.0, left_peak + right_peak - 2 * float(value))})
    return pd.DataFrame(rows)


DEPTH_RADIUS = 3  # 점수 배열에서 좌우 몇 쌍까지 볼지
scores_df = adjacent_scores(embeddings, windows, DEPTH_RADIUS)
display(scores_df.head(10))

## 4. 후보 선택 방법과 설정

| 방법 | 선택 기준 | 조절값 / 해석 |
|---|---|---|
| `threshold` | 유사도가 고정 값 이하 | `SIMILARITY_THRESHOLD`, 모델마다 분포가 달라 조정 필요 |
| `quantile` | 유사도가 하위 분위수 이하 | `LOW_QUANTILE`, 절대 유사도가 높아도 일부를 선택할 수 있음 |
| `top_k` | 유사도가 낮은 순서로 최대 K개 | `TOP_K`, 실제 전환이 없어도 후보를 요구하는 방식 |
| `local_minima` | 앞뒤보다 낮은 국소 골짜기 중 하위 분위수 | 완전히 평평한 구간과 점수 배열의 양 끝은 제외 |
| `depth` | 국소 골짜기이면서 주변 대비 하락이 큼 | `MIN_DEPTH`, `DEPTH_RADIUS` |

`MIN_DISTANCE_CHARS`는 후보 사이와 본문 양 끝에서 확보할 최소 문자 간격입니다. 유사도가 낮은 후보부터 선택하고, `depth` 방식에서는 하락 깊이가 큰 후보를 우선합니다. 따라서 Top-K도 간격 필터 뒤에는 K개보다 적을 수 있습니다. 모든 점수가 사실상 같으면 모든 방법에서 후보를 만들지 않습니다.

이 설정은 **문자 단위 후보 간격**입니다. 발화 개수를 기준으로 청크를 만들지 않습니다. 국소 골짜기 방식은 비교점이 3개 미만이면 선택할 수 없습니다.

In [ ]:
SELECTED_METHOD = "depth"  # threshold / quantile / top_k / local_minima / depth
SIMILARITY_THRESHOLD = 0.50
LOW_QUANTILE = 0.20
TOP_K = 5
MIN_DEPTH = 0.05
MIN_DISTANCE_CHARS = 400
CONTEXT_CHARS = 300  # 후보 원문 미리보기의 좌우 길이


def select_candidates(scores, method, body_start, body_end, threshold=0.5, quantile=0.2,
                      top_k=5, min_depth=0.05, min_distance_chars=400):
    methods = {"threshold", "quantile", "top_k", "local_minima", "depth"}
    if method not in methods:
        raise ValueError(f"알 수 없는 선택 방법: {method}")
    if (not np.isfinite(threshold) or not -1 <= threshold <= 1
            or not 0 < quantile < 1 or type(top_k) is not int or top_k < 0
            or not np.isfinite(min_depth) or min_depth < 0
            or type(min_distance_chars) is not int or min_distance_chars < 0):
        raise ValueError("후보 선택 설정 범위를 확인하세요.")
    s = scores["similarity"].to_numpy()
    if len(s) == 0 or float(np.ptp(s)) <= 1e-8:
        return scores.iloc[:0].copy()
    cutoff = float(np.quantile(s, quantile))
    minima = np.zeros(len(s), dtype=bool)
    if len(s) >= 3:
        minima[1:-1] = ((s[1:-1] <= s[:-2]) & (s[1:-1] <= s[2:])
                       & ((s[1:-1] < s[:-2]) | (s[1:-1] < s[2:])))
    masks = {
        "threshold": s <= threshold, "quantile": s <= cutoff,
        "top_k": np.ones(len(s), dtype=bool), "local_minima": minima & (s <= cutoff),
        "depth": minima & (scores["depth"].to_numpy() >= min_depth),
    }
    pool = scores.loc[masks[method]]
    pool = pool.sort_values(["depth", "similarity", "pair_index"], ascending=[False, True, True]) if method == "depth" else pool.sort_values(["similarity", "pair_index"])
    chosen, positions = [], []
    for idx, row in pool.iterrows():
        if method == "top_k" and len(chosen) >= top_k:
            break
        pos = int(row["candidate_char"])
        if (body_start < pos < body_end and min(pos - body_start, body_end - pos) >= min_distance_chars
                and all(abs(pos - other) >= min_distance_chars for other in positions)):
            chosen.append(idx)
            positions.append(pos)
    return scores.loc[chosen].sort_values("candidate_char").copy()


METHODS = ("threshold", "quantile", "top_k", "local_minima", "depth")
selections = {method: select_candidates(scores_df, method, body_start, body_end,
              SIMILARITY_THRESHOLD, LOW_QUANTILE, TOP_K, MIN_DEPTH, MIN_DISTANCE_CHARS) for method in METHODS}
if SELECTED_METHOD not in selections:
    raise ValueError("SELECTED_METHOD를 다섯 가지 방법 중에서 선택하세요.")
candidates_df = selections[SELECTED_METHOD]
display(pd.DataFrame([{"method": method, "candidate_count": len(frame),
                      "candidate_chars": frame["candidate_char"].tolist()} for method, frame in selections.items()]))
print("선택한 방법:", SELECTED_METHOD)
display(candidates_df)

## 5. 그래프와 원문 확인

위쪽은 인접 코사인 유사도, 아래쪽은 하락 깊이입니다. 주황색 선은 선택한 방법의 후보이며 모든 위치는 원문 문자 좌표입니다. 후보 0개도 정상적인 결과입니다. 마지막 짧은 윈도우는 입력 길이가 달라 점수에 영향을 줄 수 있으므로 원문을 함께 살펴보세요.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(scores_df["candidate_char"], scores_df["similarity"], marker=".")
axes[0].axhline(SIMILARITY_THRESHOLD, linestyle=":", color="gray", label="threshold")
axes[0].set(ylabel="Cosine similarity", title=f"{manifest['embedding_info']['model']} / {SELECTED_METHOD}")
axes[1].plot(scores_df["candidate_char"], scores_df["depth"], marker=".")
axes[1].set(xlabel="Character offset in source.txt", ylabel="Local depth")
for ax in axes:
    for pos in candidates_df["candidate_char"]:
        ax.axvline(pos, color="darkorange", alpha=0.6)
fig.tight_layout()
plt.show()

for row in candidates_df.head(10).itertuples():
    pos = int(row.candidate_char)
    print(f"\n=== 문자 {pos} / 유사도 {row.similarity:.4f} / depth {row.depth:.4f} ===")
    print(source_text[max(body_start, pos - CONTEXT_CHARS):pos])
    print("\n--- 후보 대표 위치 ---\n")
    print(source_text[pos:min(body_end, pos + CONTEXT_CHARS)])

## 6. 후보와 비교 결과 저장

`outputs/boundary-candidates/<실행폴더>/`에 설정·전체 점수·방법별 후보를 저장합니다. `candidates.json`은 선택한 방법의 후보만, `result.json`은 모든 방법을 포함합니다. 정답 경계나 성능 지표를 측정한 결과는 아닙니다. 03번은 이 `candidates.json`을 읽고 각 후보를 중심으로 좌우 원문 문맥을 LLM에 제공해 경계 여부를 검토합니다.

In [ ]:
run_id = datetime.now(timezone(timedelta(hours=9))).strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:8]
output_dir = OUTPUT_ROOT / "boundary-candidates" / run_id
output_dir.mkdir(parents=True, exist_ok=False)
settings = {"embedding_run_dir": str(EMBEDDING_RUN_DIR.resolve()),
            "embedding_manifest_sha256": hashlib.sha256((EMBEDDING_RUN_DIR / "manifest.json").read_bytes()).hexdigest(),
            "normalized_source_sha256": manifest["normalized_source_sha256"],
            "is_partial": manifest["is_partial"], "body_start_char": body_start, "body_end_char": body_end,
            "selected_method": SELECTED_METHOD, "depth_radius": DEPTH_RADIUS,
            "similarity_threshold": SIMILARITY_THRESHOLD, "low_quantile": LOW_QUANTILE,
            "top_k": TOP_K, "min_depth": MIN_DEPTH, "min_distance_chars": MIN_DISTANCE_CHARS,
            "position_rule": "midpoint of adjacent window overlap; source.txt offsets"}
result = {"status": "candidates_only", "settings": settings,
          "scores": scores_df.to_dict(orient="records"),
          "selections": {method: frame.to_dict(orient="records") for method, frame in selections.items()}}
(output_dir / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
(output_dir / "candidates.json").write_text(json.dumps({"settings": settings,
    "candidates": candidates_df.to_dict(orient="records")}, ensure_ascii=False, indent=2), encoding="utf-8")
scores_df.to_csv(output_dir / "scores.csv", index=False, encoding="utf-8-sig")
candidates_df.to_csv(output_dir / "candidates.csv", index=False, encoding="utf-8-sig")
fig.savefig(output_dir / "similarity.png", dpi=150)
print("저장 위치:", output_dir)